In [3]:
import torch
import torch.nn as nn 
import torch.optim as optim
from torchvision import datasets, transforms

# Loading MNIST DATASET

In [4]:
transform  = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,),(0.3082,))
])

In [5]:
train_dataset  = datasets.MNIST('./data', train=True , download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False , transform=transform)

100%|██████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [00:37<00:00, 266kB/s]
100%|█████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 94.5kB/s]
100%|██████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:07<00:00, 231kB/s]
100%|█████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 6.90MB/s]


In [6]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}")

Training samples: 60000
Test samples: 10000
Image shape: torch.Size([1, 28, 28])


# CNN CLass 

In [14]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)  # 1 channel in, 32 filters out
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1) # 32 channels in, 64 filters out
        self.pool = nn.MaxPool2d(2, 2)                            # halves spatial dimensions
        self.relu = nn.ReLU()
        
        # Fully connected layers
        self.fc1 = nn.Linear(64 * 7 * 7, 128)  # after 2 pooling ops: 28->14->7
        self.fc2 = nn.Linear(128, 10)           # 10 digit classes
        self.dropout = nn.Dropout(0.25)
    
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))  # conv -> relu -> pool
        x = self.pool(self.relu(self.conv2(x)))  # conv -> relu -> pool
        x = x.view(-1, 64 * 7 * 7)              # flatten for FC layers
        x = self.relu(self.fc1(self.dropout(x)))
        x = self.fc2(x)
        return x

model = CNN()
print(model)

CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
  (dropout): Dropout(p=0.25, inplace=False)
)


# Training the model 

In [15]:
# defining loss criteria and optimizer 

criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [16]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train(model, train_loader, criterion, optimizer, epoch):
    model.train()
    running_loss = 0.0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch}, Average Loss: {avg_loss:.4f}")

def evaluate(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in test_loader:
            output = model(data)
            _, predicted = torch.max(output, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    
    print(f"Test Accuracy: {100 * correct / total:.2f}%")

# Train for 5 epochs
for epoch in range(1, 6):
    train(model, train_loader, criterion, optimizer, epoch)
    evaluate(model, test_loader)

Epoch 1, Average Loss: 0.1393
Test Accuracy: 98.50%
Epoch 2, Average Loss: 0.0497
Test Accuracy: 98.87%
Epoch 3, Average Loss: 0.0370
Test Accuracy: 99.15%
Epoch 4, Average Loss: 0.0270
Test Accuracy: 99.12%
Epoch 5, Average Loss: 0.0221
Test Accuracy: 99.17%
